In [1]:
import zipfile
import os
import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
import optuna # <-- NUEVO: Importar optuna
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import KFold
zip_file_path = '../data/MeIA2025-Reto-01.zip'

extracted_folder_path = '../data/extracted_corpus/'

# 2. Crear la carpeta de extracción si no existe
if not os.path.exists(extracted_folder_path):
    os.makedirs(extracted_folder_path)
    print(f"Carpeta '{extracted_folder_path}' creada.")
# 3. Descomprimir archivo  
try:
    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
        zip_ref.extractall(extracted_folder_path)
    print(f"'{zip_file_path}' descomprimido exitosamente en '{extracted_folder_path}'.")
except FileNotFoundError:
    print(f"Error: El archivo ZIP no se encontró en '{zip_file_path}'. Verifica la ruta.")
except Exception as e:
    print(f"Ocurrió un error al descomprimir el archivo: {e}")
    
# 4. Listar los archivos descomprimidos (para verificar)
print("\nArchivos en la carpeta del corpus:")
corpus_files = os.listdir(extracted_folder_path)
for file_name in corpus_files:
    print(f"- {file_name}")

# Definir la ruta base donde se extrajo el contenido del ZIP
# Asegúrate de que esta ruta sea correcta relativa a tu notebook test.ipynb
# Si tu notebook está en 'notebooks/' y la extracción está en 'data/extracted_corpus/Datos-MelA-Reto-01/'
base_extracted_path = '../data/extracted_corpus/Datos-MeIA-Reto-01/'

# Rutas completas a los archivos XLSX
train_file_path = os.path.join(base_extracted_path, 'MeIA_2025_train.xlsx')
test_file_path = os.path.join(base_extracted_path, 'MeIA_2025_test_wo_labels.xlsx')

print(f"Intentando cargar el archivo de entrenamiento desde: {train_file_path}")
print(f"Intentando cargar el archivo de prueba desde: {test_file_path}")

try:
    # Cargar el dataset de entrenamiento
    
    df_train = pd.read_excel(train_file_path)
    print(f"Datos de entrenamiento cargados correctamente")
    # Cargar el dataset de prueba (sin etiquetas)
    df_test = pd.read_excel(test_file_path)
    print(f"Datos de test cargados correctamente")


except FileNotFoundError:
    print(f"Error: Uno de los archivos XLSX no se encontró.")
    print(f"Asegúrate de que las rutas sean correctas: '{train_file_path}' y '{test_file_path}'")
    print(f"Y que la carpeta 'Datos-MelA-Reto-01' esté dentro de 'extracted_corpus'.")
except Exception as e:
    print(f"Ocurrió un error al cargar los archivos Excel: {e}")


2025-06-17 17:24:00.844755: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1750191841.309255  100755 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1750191841.500206  100755 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1750191842.555445  100755 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1750191842.555493  100755 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1750191842.555494  100755 computation_placer.cc:177] computation placer alr

'../data/MeIA2025-Reto-01.zip' descomprimido exitosamente en '../data/extracted_corpus/'.

Archivos en la carpeta del corpus:
- Datos-MeIA-Reto-01
Intentando cargar el archivo de entrenamiento desde: ../data/extracted_corpus/Datos-MeIA-Reto-01/MeIA_2025_train.xlsx
Intentando cargar el archivo de prueba desde: ../data/extracted_corpus/Datos-MeIA-Reto-01/MeIA_2025_test_wo_labels.xlsx
Datos de entrenamiento cargados correctamente
Datos de test cargados correctamente


In [2]:
print("Creando input estructurado...")
df_train['structured_text'] = df_train.apply(
    lambda row: f"tipo: {str(row['Type']).lower()}. pueblo: {str(row['Town']).lower()}. reseña: {str(row['Review']).lower()}",
    axis=1
)

Creando input estructurado...


In [4]:
from sklearn.model_selection import KFold
df_model = df_train.rename(columns={'structured_text': 'text', 'Polarity': 'label'})
df_model = df_model[['text', 'label']].copy()
df_model['label'] = df_model['label'].apply(lambda x: int(x) - 1)


# --- 2. Modelo, Tokenizador y Métricas ---
model_name = "tabularisai/multilingual-sentiment-analysis"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def model_init():
    return AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=5)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=512)

# La métrica oficial del reto es F1-Macro
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {"f1_macro": f1_score(labels, predictions, average="macro")}


# --- 3. El Loop de K-Fold ---
N_SPLITS = 5
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42) # Usar random_state para resultados reproducibles
all_scores = []

print(f"Iniciando entrenamiento con K-Fold de {N_SPLITS} splits...")

for fold, (train_idx, val_idx) in enumerate(kf.split(df_model)):
    print(f"\n===== FOLD {fold+1}/{N_SPLITS} =====")
    
    # Crear datasets para este fold específico
    train_df = df_model.iloc[train_idx]
    eval_df = df_model.iloc[val_idx]
    
    train_dataset = Dataset.from_pandas(train_df)
    eval_dataset = Dataset.from_pandas(eval_df)

    tokenized_train_dataset = train_dataset.map(tokenize_function, batched=True)
    tokenized_eval_dataset = eval_dataset.map(tokenize_function, batched=True)

    # Directorio de salida único para cada fold
    output_dir = f".modelos_tabu/kfold_model_fold_{fold+1}"

    training_args = TrainingArguments(
        output_dir=output_dir,
        
        # --- PARÁMETROS FIJOS (LOS QUE ELEGIMOS) ---
        learning_rate=2e-5, # El valor por defecto, lo hacemos explícito
        num_train_epochs=3,
        per_device_train_batch_size=16,
        warmup_steps=100,
        weight_decay=0.01,
        
        # --- Argumentos de logística ---
        logging_strategy="epoch",
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,
        report_to="none",
    )

    trainer = Trainer(
        model_init=model_init,
        args=training_args,
        train_dataset=tokenized_train_dataset,
        eval_dataset=tokenized_eval_dataset,
        tokenizer=tokenizer,
        compute_metrics=compute_metrics,
    )
    
    trainer.train()
    
    eval_results = trainer.evaluate()
    score = eval_results["eval_f1_macro"]
    all_scores.append(score)
    print(f"F1-Macro para el Fold {fold+1}: {score:.4f}")

    # Guardamos el modelo final del fold
    trainer.save_model(output_dir)

# --- 4. Resultados Finales de la Validación ---
print("\n===== Resultados K-Fold =====")
print(f"F1-Macro scores por fold: { [round(s, 4) for s in all_scores] }")
print(f"F1-Macro Promedio: {np.mean(all_scores):.4f}")
print(f"Desviación Estándar: {np.std(all_scores):.4f}")


Iniciando entrenamiento con K-Fold de 5 splits...

===== FOLD 1/5 =====


Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

/tmp/ipykernel_100755/950079555.py:68: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,F1 Macro
1,1.138300,1.002343,0.550528
2,0.951000,1.010575,0.546448
3,0.800400,1.021025,0.575130


...

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

/tmp/ipykernel_100755/950079555.py:68: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,F1 Macro
1,1.134800,1.067716,0.521658
2,0.934200,1.062525,0.509242
3,0.763700,1.120162,0.510611


F1-Macro para el Fold 5: 0.5217

===== Resultados K-Fold =====
F1-Macro scores por fold: [0.5751, 0.5517, 0.5356, 0.5062, 0.5217]
F1-Macro Promedio: 0.5381
Desviación Estándar: 0.0239
